# Использование архитектуры Transformer для классификации текстов

## Работа в аудитории

Рассмотрим дообучение предварительно обученной модели BERT для анализа тональности с помощью популярного набора данных IMDb sentiment.

In [ ]:
!pip install -q transformers[torch] datasets accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

По возможности подключим графический процессор:

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

Мы будем использовать здесь класс DistilBertForSequenceClassification, который унаследован от класса DistilBert со специальным дополнением для классификации последовательностей поверх основного кода. Мы можем использовать этот классификатор для обучения моделей классификации, у которых количество классов по умолчанию равно 2:

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

model_path= 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(
    model_path,
    id2label={0: "NEG", 1: "POS"},
    label2id={"NEG": 0, "POS": 1}
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Обратим внимание, что два параметра, называемых `id2label` и `label2id`, передаются модели для использования во время вывода. В качестве альтернативы мы можем создать конкретный объект `config`.

Теперь скачаем набор данных классификации тональности под названием IMDB Dataset.
Исходный набор данных состоит из двух наборов данных: 25 000 примеров для обучения и 25 примеров для тестирования. Мы разделим набор данных на два блока: тестовый и проверочный.
Обратим внимание, что примеры для первого блока положительны, а примеры второго блока – отрицательны.
Распределим примеры следующим образом:

In [ ]:
from datasets import load_dataset

imdb_train= load_dataset('imdb', split="train[:2000]+train[-2000:]")
imdb_test= load_dataset('imdb', split="test[:500]+test[-500:]")
imdb_val= load_dataset('imdb', split="test[500:1000]+test[-1000:-500]")

Проверим форму набора данных:

In [ ]:
imdb_train.shape, imdb_test.shape, imdb_val.shape

((4000, 2), (1000, 2), (1000, 2))

Теперь пропустим эти наборы данных через модель tokenizer, чтобы подготовить их к обучению:

In [ ]:
max_length = 256
enc_train = imdb_train.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length), batched=True, batch_size=1000)
enc_test =  imdb_test.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length), batched=True, batch_size=1000)
enc_val =   imdb_val.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length), batched=True, batch_size=1000)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Посмотрим, как выглядит тренировочный набор. Токенизатор добавил в набор данных маску внимания и входные идентификаторы, чтобы модель BERT могла его обрабатывать:

In [ ]:
import pandas as pd
pd.DataFrame(enc_train).head()

,text,label,input_ids,attention_mask
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,"[101, 1045, 12524, 1045, 2572, 8025, 1011, 375...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"""I Am Curious: Yellow"" is a risible and preten...",0,"[101, 1000, 1045, 2572, 8025, 1024, 3756, 1000...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,If only to avoid making this type of film in t...,0,"[101, 2065, 2069, 2000, 4468, 2437, 2023, 2828...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,This film was probably inspired by Godard's Ma...,0,"[101, 2023, 2143, 2001, 2763, 4427, 2011, 2643...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,"Oh, brother...after hearing about this ridicul...",0,"[101, 2821, 1010, 2567, 1012, 1012, 1012, 2044...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


На этом этапе наборы данных готовы для обучения и тестирования. Класс Trainer и класс TrainingArguments позволяют преодолеть большую часть сложностей обучения.
Мы определим набор аргументов в классе TrainingArguments, который затем будет передан объекту Trainer. Он включает следующие параметры:

|Параметр|Определение|
|---|---|
|output_dir|Указатель на каталог, в котором будут сохранены контрольные точки модели и прогнозы|
|do_train и do_eval|Опции для мониторинга производительности модели в процессе обучения|
|logging_strategy|Возможные опции ведения лога: отключено, эпохи или шаги (по умолчанию)|
|logging_steps (по умолчанию 500)|Количество шагов между двумя записями в лог, которые будут сохранены в каталог logging_dir|
|save_strategy|Опции сохранения контрольных точек: отключено, эпохи и шаги (по умолчанию)|
|save_steps (по умолчанию 500)|Количество шагов между контрольными точками|
|fp16|Указывает на использование смешанной точности и использу- ет как 16-, так и 32-битные типы с плавающей запятой, чтобы обучение проходило быстрее и занимало меньше памяти|
|load_best_model_at_end|После окончания обучения загружает из контрольной точки модель с наилучшим качеством|
|logging_dir|Каталог дла сохранения логов TensorBoard|


In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir='./BinClassificationModel',
    do_train=True,
    do_eval=True,
    num_train_epochs=3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_strategy='steps',
    logging_dir='./logs',
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=cuda.is_available(),
    load_best_model_at_end=True,
    report_to='none'
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Хотя архитектуры глубокого обучения, такие как LSTM, требуют для обучения много эпох, иногда более 50, для тонкой настройки модели на основе трансформера нам будет достаточно эпохи номер 3 из-за переноса обучения. В большинстве случаев трех эпох достаточно для тонкой настройки, поскольку предварительно обученная модель тщательно изучает язык на этапе предварительного обучения, который в среднем занимает около 50 эпох.

Далее можно увидеть, что 3-х эпох бывает достаточно для решения многих последующих задач.
В процессе обучения контрольные точки модели будут сохраняться в каталоге ./BinClassificationModel через каждые 200 шагов.

Перед созданием экземпляра объекта Trainer определим метод `compute_metrics()`, который позволит отслеживать ход обучения с точки зрения конкретных показателей:

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, acc = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall
    }

Далее создадим и запустим объект Trainer (это оптимизированный инструмент для организации сложных процессов обучения и оценки моделей машинного обучения):

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=enc_train,
    eval_dataset=enc_val,
    compute_metrics=compute_metrics
)

Приступим к процессу обучения. В ходе выполнения следующего кода будет выполняться регистрация целевых показателей качества (Accuracy, F1 и др.):

In [ ]:
results = trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1,Precision,Recall
1,No log,0.649626,0.002600,0.737000,0.736808,0.737693,0.737000
2,0.619700,0.356161,0.002600,0.833000,0.832105,0.840253,0.833000
3,0.619700,0.360775,0.002600,0.853000,0.851930,0.863505,0.853000


В конце обучения объект Trainer сохраняет контрольную точку, потеря валидации которой является наименьшей.
Проверим качество лучшей контрольной точки на трех наборах данных (обучение/тестирование/валидация):

In [ ]:
q=[trainer.evaluate(eval_dataset=data) for data in [enc_train, enc_val, enc_test]]
pd.DataFrame(q, index=["train","val","test"]).iloc[:,:5]

,eval_loss,eval_model_preparation_time,eval_Accuracy,eval_F1,eval_Precision
train,0.283454,0.0026,0.890,0.889533,0.896704
val,0.356161,0.0026,0.833,0.832105,0.840253
test,0.338030,0.0026,0.856,0.854716,0.869043


Результаты показывают успешное качество модели на валидационной и тестовой выборках.

Теперь воспользуемся моделью, чтобы проверить, правильно ли она работает.
Сначала сохраним модель и токенизатор:

In [ ]:
model_save_path = "BestBinClassificationModel"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

('BestBinClassificationModel/tokenizer_config.json',
 'BestBinClassificationModel/special_tokens_map.json',
 'BestBinClassificationModel/vocab.txt',
 'BestBinClassificationModel/added_tokens.json',
 'BestBinClassificationModel/tokenizer.json')

Чтобы упростить шаги предсказания определим следующую функцию:

In [ ]:
def get_prediction(text):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=250, return_tensors="pt").to(device)
    outputs = model(inputs["input_ids"].to(device),inputs["attention_mask"].to(device))
    probs = outputs[0].softmax(1)
    return probs, probs.argmax()

Теперь запустим модель для проверки:

In [ ]:
text = "I didn't like the movie since it bored me"
get_prediction(text)[1].item()

0

Здесь 0 означает отрицательный вывод.
Поскольку выше уже был определен идентификатор  метки, то используем функцию `pipeline` работы всей схемы модели, включая получение меток:

In [ ]:
from transformers import pipeline, DistilBertForSequenceClassification, DistilBertTokenizerFast
model = DistilBertForSequenceClassification.from_pretrained("BestBinClassificationModel")
tokenizer= DistilBertTokenizerFast.from_pretrained("BestBinClassificationModel")
nlp= pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Device set to use cuda:0


Введем позитивный текст:

In [ ]:
nlp("the movie was very impressive")

[{'label': 'POS', 'score': 0.8848980665206909}]

Введем негативный текст:

In [ ]:
nlp("the script of the picture was very poor")

[{'label': 'NEG', 'score': 0.8800569772720337}]

Таким образом мы успешно провели дообучение языковой модели. Целевые метрики и частные примеры подтвердили практическую значимость модели.

## Самостоятельная работа

**Задание.** Используя набор данных, который был использован при выполенении [самостоятельной работы](https://colab.research.google.com/drive/1iRIsauho2O9XSWKOoAR-4UczvYwmQbD5?usp=sharing), обучите модель классификации с использованием архитектуры Transformer.

Ваша задача - обучить модель, которая получит на тестовой части качество по метрике accuracy более 0.9.

In [ ]:
!pip install transformers >> None

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from transformers import TrainingArguments, Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
model_path = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(
    model_path,
    id2label={0: "NEG", 1: "POS"},
    label2id={"NEG": 0, "POS": 1}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
df = pd.read_csv("https://code.s3.yandex.net/datasets/tweets_lemm_train.csv")
df.head()

,text,positive,lemm_text
0,"@first_timee хоть я и школота, но поверь, у на...",1,хоть я и школотый но поверь у мы то же самый о...
1,"Да, все-таки он немного похож на него. Но мой ...",1,да весь таки он немного похожий на он но мой м...
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,ну ты идиотка я испугаться за ты
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,кто то в угол сидеть и погибать от голод а мы ...
4,@irina_dyshkant Вот что значит страшилка :D\r\...,1,вот что значит страшилка но блин посмотреть ве...


In [ ]:
df = df.rename(columns={'positive': 'label'})[['text', 'label']]
df.shape

(5000, 2)

In [ ]:
df_train, test_eval = train_test_split(df, test_size=0.2)
df_val, df_test = train_test_split(test_eval, test_size=0.5)

In [ ]:
ds_train = Dataset.from_pandas(df_train)
ds_test = Dataset.from_pandas(df_test)
ds_val = Dataset.from_pandas(df_val)

max_length = 256
batch_size = 32

enc_train = ds_train.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length),
                                             batched=True, batch_size=batch_size)
enc_test = ds_test.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length),
                                           batched=True, batch_size=batch_size)
enc_val = ds_val.map(lambda e: tokenizer(e['text'], padding='max_length', truncation=True, max_length=max_length),
                                         batched=True, batch_size=batch_size)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
pd.DataFrame(enc_train).head()

,text,label,__index_level_0__,input_ids,attention_mask
0,Трудовыебудни во славу сатане(((((((((((((,0,3657,"[101, 1197, 16856, 29748, 29742, 19259, 29113,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"@MrKurumo я не вдруг!( просто люблю вас, вы пр...",0,4855,"[101, 1030, 2720, 18569, 2819, 2080, 1210, 119...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,RT @Prosto_olen: Скоро я составлю список перло...,0,4075,"[101, 19387, 1030, 4013, 16033, 1035, 15589, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,@KnyazhevaMasha завтра пойдешь на олимпиаду? И...,0,4565,"[101, 1030, 14161, 3148, 27922, 13331, 9335, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,@khonmaria хахахахаха в меру испорченности:D\r...,1,2474,"[101, 1030, 1047, 8747, 7849, 2401, 1200, 1026...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
training_args = TrainingArguments(
    output_dir='./BinClassificationModel',
    do_train=True,
    do_eval=True,
    num_train_epochs=3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_strategy='steps',
    logging_dir='./logs',
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=cuda.is_available(),
    load_best_model_at_end=True,
    report_to='none'
)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, acc = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=enc_train,
    eval_dataset=enc_val,
    compute_metrics=compute_metrics
)

In [ ]:
results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.434300,0.258447,0.946000,0.945922,0.948179,0.945863
2,0.014700,0.005995,1.000000,1.000000,1.000000,1.000000
3,0.008300,0.003661,0.998000,0.998000,0.998000,0.998008


In [ ]:
q = [trainer.evaluate(eval_dataset=data) for data in [enc_train, enc_val, enc_test]]
pd.DataFrame(q, index=["train", "val", "test"])

,eval_loss,eval_Accuracy,eval_F1,eval_Precision,eval_Recall,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
train,0.002219,0.99975,0.99975,0.999749,0.999751,9.1590,436.730,6.878,3.0
val,0.003661,0.99800,0.99800,0.998000,0.998008,1.0558,473.586,7.577,3.0
test,0.001576,1.00000,1.00000,1.000000,1.000000,1.0642,469.840,7.517,3.0
